# Chapter 14 — Multi-Agent: When, How and When Not To

*The decision tree starts with: don't, unless.*

## Objective

Build a tiny supervisor with two workers. Each worker has its own `GovernanceHarness` and therefore its own audit chain. Then list the questions to answer before reaching for multi-agent.

In [1]:
from agentlab.core import BaseAgent, Finish
from agentlab.governance import GovernanceHarness
from agentlab.multiagent import MessageBus, Supervisor, Worker
from agentlab.tools import GovernedToolExecutor, ToolRegistry

## Build two workers, each with its own harness

In [2]:
class FixedFinish(BaseAgent):
    def __init__(self, answer): self.answer = answer
    def propose_action(self, state):
        return Finish(output=self.answer)

def make_worker(name, capability, answer):
    executor = GovernedToolExecutor(ToolRegistry(), gates=[])
    harness = GovernanceHarness(FixedFinish(answer), executor)
    return Worker(name=name, capability=capability, harness=harness)

classifier = make_worker('classifier', 'classify_complaint', 'complaint')
drafter    = make_worker('drafter',    'draft_response',     {'text': 'thank you for reaching out'})

## A supervisor routes by name

Messages are typed `AgentMessage` objects on a shared `MessageBus`. Free-form chatter is the failure mode this module is designed to prevent.

In [3]:
bus = MessageBus()
sup = Supervisor(name='sup', workers=[classifier, drafter], bus=bus)

r1 = sup.delegate('classifier', goal='classify this message')
r2 = sup.delegate('drafter',    goal='draft a response')
print('classifier response:', r1.payload)
print('drafter response:   ', r2.payload)
print(f'bus has {len(bus)} messages')

classifier response: {'status': 'done', 'final_output': 'complaint', 'steps': 1}
drafter response:    {'status': 'done', 'final_output': {'text': 'thank you for reaching out'}, 'steps': 1}
bus has 4 messages


## Each worker has an independent audit chain

This is the governance-per-worker guarantee. If `classifier` is later replaced, its history is preserved separately.

In [4]:
print('classifier audit verifies:', classifier.harness.audit.verify())
print('drafter audit verifies:   ', drafter.harness.audit.verify())
print('classifier chain head[:8]:', classifier.harness.audit.head()[:8])
print('drafter chain head[:8]:   ', drafter.harness.audit.head()[:8])

classifier audit verifies: True
drafter audit verifies:    True
classifier chain head[:8]: 104e2a30
drafter chain head[:8]:    386df6c0


## When *not* to use multi-agent

Multi-agent is often where systems go wrong. Before reaching for it, answer all three:

1. Can a single well-engineered agent with planning + tools do this?
2. Do the workers really have different capabilities, policies or audit boundaries?
3. Are you prepared for the extra latency, the extra failure surface and the harder evaluation?

If the answer to (1) is yes, stop. If (2) is no, stop. Multi-agent buys you parallel specialization and clean governance boundaries. It does **not** buy you intelligence.

## Conflict resolution by evidence, not by vote

Correlated workers (same base model) can form a two-to-one majority of correlated errors. Instead of voting, the supervisor scores each worker's claim against the shared substrate with `score_triple` and keeps the better-grounded one — the substrate acting as **orchestrator**.

In [5]:
import torch
from pathlib import Path
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore
from agentlab.gms_backend import GMSMemory

_root = Path('.') if Path('data/gms_banking_store').exists() else Path('..')
store = GMSExpertStore(
    DocGMSConfig(store_path=str(_root / 'data' / 'gms_banking_store')),
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
)
store.load()
mem = GMSMemory(store)

claims = {'worker_A': ('overdraft', 'has_fee_amount', '35.0'),
          'worker_B': ('overdraft', 'has_fee_amount', '45.0')}
scored = {w: mem.score_triple(*c) for w, c in claims.items()}
winner = min(scored, key=scored.get)
for w, s in scored.items():
    print(f'  {w}: {claims[w][2]:>6}  groundedness={s:.3f}')
print(f'supervisor accepts {winner} by evidence (not by vote)')

knowlytix-core v0.2.0 licensed to customer=KnowlytixAgentBuilder tier=enterprise expires=2027-06-04


  GMS entities:  81
  GMS relations: 17
  GMS triples:   116


/path/to/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


  Store loaded from data/gms_banking_store
  Entities:  81
  Relations: 17
  Triples:   116
  ENM:       15
  Documents: 1


  worker_A:   35.0  groundedness=0.865
  worker_B:   45.0  groundedness=1.599
supervisor accepts worker_A by evidence (not by vote)


## Anti-patterns flagged here

- Reaching for multi-agent before single-agent is exhausted.
- Free-form agent chatter.
- Voting as a substitute for evidence.

In [6]:
# Self-check
assert r1.payload['final_output'] == 'complaint'
assert classifier.harness.audit is not drafter.harness.audit
# GMS arbitration keeps the grounded claim, not the correlated majority.
assert winner == 'worker_A'
assert scored['worker_A'] < scored['worker_B']
print('OK')

OK
